# LangChain L14 — Level 13 — Streaming, observability and the production shape
OpsPilot now has tools, memory, knowledge, guardrails, approvals and specialists. Three
operational concerns remain before it can face users: **streaming** (nobody waits 30 seconds
staring at a spinner), **observability** ("why did the agent refund this customer?") and
**cost** (four model calls per request at 100,000 requests a month is real money).

Finally we assemble every layer into one agent and look at the shape of the whole system.

### Step 1 — Streaming progress and tokens

`stream(stream_mode="updates")` yields one item per node as it finishes: the UI can show
"looking up order..." while the tools run. `stream_mode="messages"` yields model tokens as they
are generated. Both work on any agent or graph in this notebook.

In [ ]:
print("UPDATES (one per node):")
for update in opspilot_rag.stream({"messages": [{"role": "user", "content": "What is the weather in London, and what is the shipping policy for damaged deliveries?"}]}, stream_mode="updates"):
    for node, payload in update.items():
        last = payload["messages"][-1] if isinstance(payload, dict) and payload.get("messages") else None
        summary = (", ".join(c["name"] for c in last.tool_calls) if isinstance(last, AIMessage) and last.tool_calls else text_of(last)[:70]) if last else ""
        print(f"  {node:6} -> {summary}")

print("\nTOKENS (from the model node only):")
for token, metadata in opspilot_rag.stream({"messages": [{"role": "user", "content": "In one sentence, what does OpsPilot do?"}]}, stream_mode="messages"):
    if metadata.get("langgraph_node") == "model" and text_of(token):
        print(text_of(token), end="", flush=True)
print()

### Step 2 — A trace: what actually happened, and what it cost

A class-based middleware records every model call and tool call with timings, and sums the
token usage. This is a hand-made trace; **LangSmith** does the same automatically for every
run once two environment variables are set (`LANGSMITH_TRACING=true`, `LANGSMITH_API_KEY=...`),
and adds a UI to browse them. Either way, the questions you can now answer are the ones that
matter in production: which tool was chosen, with what arguments, how long it took, what it cost.

In [ ]:
from langchain.agents.middleware import AgentMiddleware

class TraceMiddleware(AgentMiddleware):
    """Records model and tool calls into self.events; sums token usage."""
    def __init__(self):
        super().__init__()
        self.events, self.tokens = [], {"input": 0, "output": 0}

    def wrap_model_call(self, request, handler):
        started = time.perf_counter()
        response = handler(request)
        usage = response.result[0].usage_metadata or {}
        self.tokens["input"] += usage.get("input_tokens", 0); self.tokens["output"] += usage.get("output_tokens", 0)
        self.events.append(("model", f"{len(request.messages)} msgs -> {len(response.result[0].tool_calls)} tool calls", round(1000 * (time.perf_counter() - started))))
        return response

    def wrap_tool_call(self, request, handler):
        started = time.perf_counter()
        response = handler(request)
        self.events.append(("tool", f"{request.tool_call['name']}({json.dumps(request.tool_call['args'])})", round(1000 * (time.perf_counter() - started))))
        return response

def cost_estimate(tokens, usd_per_million_in=0.15, usd_per_million_out=0.60):
    """Illustrative prices; check your provider's current price list."""
    return tokens["input"] / 1e6 * usd_per_million_in + tokens["output"] / 1e6 * usd_per_million_out

trace = TraceMiddleware()
traced = create_agent(model=model, tools=KNOWLEDGE_TOOLS, system_prompt=OPSPILOT_PROMPT, middleware=[trace])
out = traced.invoke({"messages": [{"role": "user", "content": "Customer C001 bought order O1001 12 days ago and wants a refund. Does the refund policy allow it for their plan?"}]})

print("TRACE:")
for kind, detail, ms in trace.events:
    print(f"  {kind:5} {ms:5d} ms  {detail[:90]}")
print(f"tokens: {trace.tokens} | est. cost per request: ${cost_estimate(trace.tokens):.5f} | per 100k requests: ${100_000 * cost_estimate(trace.tokens):,.2f}")

### Step 3 — The assembled OpsPilot

Every layer from L3 to L11 on one agent: read and write tools, policy search, long-term memory,
persistence, role-based permissions, a tool-boundary guard, call limits, retries, human approval
and the trace. The request below runs as the finance role, pauses for approval, and completes.

In [ ]:
final_trace = TraceMiddleware()
ALL_TOOLS = KNOWLEDGE_TOOLS + WRITE_TOOLS + [remember_preference, recall_preferences]

opspilot_final = create_agent(
    model=model,
    tools=ALL_TOOLS,
    system_prompt=OPSPILOT_PROMPT + " Search the policy documents before any refund. Money-moving actions are reviewed by a human.",
    middleware=[
        final_trace,                                                    # observability
        permission_filter,                                              # role decides visible tools
        block_unauthorised,                                             # tool-boundary guard
        ModelCallLimitMiddleware(run_limit=8, exit_behavior="end"),     # never loop forever
        ToolRetryMiddleware(max_retries=2, initial_delay=0.1),          # transient failures
        HumanInTheLoopMiddleware(interrupt_on={"refund_customer": True}),  # approval for writes
    ],
    checkpointer=InMemorySaver(),
    store=store,
    context_schema=Context,
)

case = {"configurable": {"thread_id": "final-1"}}
finance = Context(user_id="rahul", role="finance")
result = opspilot_final.invoke({"messages": [{"role": "user", "content": "Customer C002 was charged twice for order O1002. Check the order and the refund policy, then issue a refund of 500 to C002."}]}, case, context=finance)

if "__interrupt__" in result:
    pending = result["__interrupt__"][0].value["action_requests"]
    print("PAUSED for approval:", [(a["name"], a["args"]) for a in pending])
    result = opspilot_final.invoke(Command(resume={"decisions": [{"type": "approve"}] * len(pending)}), case, context=finance)

print("\nANSWER:", text_of(result["messages"][-1])[:160])
print("\nTRAJECTORY:")
show_messages(result["messages"])
print("\nTRACE:", [(k, d[:40], ms) for k, d, ms in final_trace.events])
print("ledger:", REFUND_LEDGER[-1])

### Step 4 — The production shape

What we built maps onto the architecture that industry agent systems converge on:

```text
                         USER
                           |
                  +-----------------+
                  | API / Frontend  |   streaming (L14)
                  +--------+--------+
                           |
                  +-----------------+
                  | Auth / Identity |   context_schema: user_id, role (L7, L10)
                  +--------+--------+
                           |
             +-----------------------------+
             |         Middleware          |
             |  permissions   (L10)        |
             |  guardrails    (L10)        |
             |  limits/retry  (L10)        |
             |  summarisation (L6)         |
             |  human approval(L11)        |
             |  tracing       (L14)        |
             +--------------+--------------+
                            |
                  +-----------------+
                  |  Agent (L3)     |   or an explicit LangGraph workflow (L12)
                  +--------+--------+
                           |
            +--------------+--------------+
            |              |              |
         Tools (L4)     RAG (L8)     Sub-agents (L13)
            |              |              |
       APIs / DBs     Vector store    Specialists

      +---------------------------+   +---------------------------+
      | Persistence: checkpointer |   | Observability: traces,    |
      | + store (L6, L7, L12)     |   | cost, evaluation (L14)    |
      +---------------------------+   +---------------------------+
```

Five disciplines hide inside "agent engineering": LLM engineering (prompts, tools, structured
output), software engineering (interfaces, validation, errors), distributed systems (retries,
idempotency, persistence), security (authorisation, injection, approval) and evaluation. The
first question to ask about any new problem remains: **should this be an agent at all?** If the
steps are known in advance, a workflow (L12) or a plain function is cheaper, faster and safer.

**Where to go next:** LangSmith evaluation datasets for trajectory and correctness checks; SQLite
or Postgres checkpointers for real persistence; `ProviderStrategy` structured output on providers
that support it; LangGraph Platform or your own FastAPI service for deployment.

### Recap

- **Problem seen:** no progress feedback, no record of what happened, no idea what a request costs.
- **Layer added:** streaming modes, a trace middleware with token accounting, and the fully assembled OpsPilot.
- **Evidence:** the final run showed every node as it finished, paused for approval, and produced a trace with a cost estimate.